# Task 4: Random Forests & Ensembles

**Name:** Nurdaulet Zhussip  
**Course:** MOP 3231

In [1]:
import platform
import math

import matplotlib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 2729

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"matplotlib: {matplotlib.__version__}")

Python: 3.12.0
NumPy: 2.3.4
pandas: 2.3.3
scikit-learn: 1.7.2
matplotlib: 3.10.7


In [2]:
tree = DecisionTreeClassifier(random_state=SEED)
bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, oob_score=True, random_state=SEED)
rf = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=SEED, n_jobs=-1)
lr = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), LogisticRegression(max_iter=2000))
bag_lr = BaggingClassifier(lr, n_estimators=50, random_state=SEED)
knn = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), KNeighborsClassifier(n_neighbors=15))
ada = AdaBoostClassifier(n_estimators=400, random_state=SEED)
members = [("lr", lr), ("knn", knn), ("tree", tree)]

In [3]:
data = pd.read_csv("task-04-ML-course/data/spambase.csv")
print(f"Dataset shape: {data.shape}")
class_balance = data["spam"].value_counts(normalize=True).sort_index()
class_balance.index.name = "spam"
class_balance.name = "share"
display(class_balance.to_frame().round(4))

Dataset shape: (4204, 58)


,share
spam,
0,0.6013
1,0.3987


In [4]:
X = data.drop(columns="spam")
y = data["spam"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=SEED,
)

print(f"Training rows: {len(X_train)}; test rows: {len(X_test)}")
print(f"Training spam share: {y_train.mean():.4f}")
print(f"Test spam share: {y_test.mean():.4f}")

Training rows: 3153; test rows: 1051
Training spam share: 0.3987
Test spam share: 0.3987


In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(f"Shared cross-validation folds: {cv.get_n_splits()}")

Shared cross-validation folds: 5


## Part 1

We will reconstruct the bagging model's out-of-bag mask and accuracy, compare all five fixed models on the same stratified folds, and compare the forest's OOB, CV, and held-out test estimates.

In [6]:
def oob_mask(sample_indices, n_samples):
    """Return a boolean mask for rows absent from a bootstrap sample."""
    mask = np.ones(n_samples, dtype=bool)
    mask[np.asarray(sample_indices, dtype=int)] = False
    return mask

In [7]:
tiny_expected = np.array([False, True, False, True, True])
tiny_result = oob_mask([0, 2, 2], 5)
empty_result = oob_mask([], 3)
array_result = oob_mask(np.array([3, 0]), 4)
print("Tiny example:", tiny_result, "matches:", np.array_equal(tiny_result, tiny_expected))
print("Empty sample:", empty_result, "matches:", np.array_equal(empty_result, [True, True, True]))
print("Array sample:", array_result, "matches:", np.array_equal(array_result, [False, True, True, False]))

Tiny example: [False  True False  True  True] matches: True
Empty sample: [ True  True  True] matches: True
Array sample: [False  True  True False] matches: True


For `oob_mask([0, 2, 2], 5)`, the result was `[False, True, False, True, True]`; it matched the expected mask. The empty-list and integer-array examples also matched (`True`).

In [8]:
bag.fit(X_train, y_train)
n_train = len(X_train)
oob_shares = np.array([
    oob_mask(indices, n_train).mean()
    for indices in bag.estimators_samples_
])
oob_formula = (1 - 1 / n_train) ** n_train
inbag_formula = 1 - oob_formula
print(f"Mean OOB share across {len(oob_shares)} trees: {oob_shares.mean():.4f}")
print(f"Mean in-bag share across trees: {1 - oob_shares.mean():.4f}")
print(f"Theoretical OOB share (1 - 1/n)^n: {oob_formula:.4f}")
print(f"Theoretical in-bag share: {inbag_formula:.4f}")

Mean OOB share across 100 trees: 0.3679
Mean in-bag share across trees: 0.6321
Theoretical OOB share (1 - 1/n)^n: 0.3678
Theoretical in-bag share: 0.6322


Across 100 trees, the mean OOB share was 0.3679 versus the theoretical 0.3678, and the mean in-bag share was 0.6321 versus 0.6322. With 3,153 training rows, the empirical and theoretical bootstrap shares closely agree.

In [9]:
votes = np.zeros((n_train, 2), dtype=int)
for estimator, sample_indices in zip(bag.estimators_, bag.estimators_samples_):
    mask = oob_mask(sample_indices, n_train)
    row_indices = np.flatnonzero(mask)
    predictions = estimator.predict(X_train.iloc[row_indices].to_numpy()).astype(int)
    votes[row_indices, predictions] += 1

own_oob_predictions = np.argmax(votes, axis=1)
own_oob_accuracy = np.mean(own_oob_predictions == y_train.to_numpy())
print(f"Own OOB accuracy: {own_oob_accuracy:.6f}")
print(f"bag.oob_score_: {bag.oob_score_:.6f}")
print("Exactly equal:", own_oob_accuracy == bag.oob_score_)
print(f"Absolute difference: {abs(own_oob_accuracy - bag.oob_score_):.12f}")

Own OOB accuracy: 0.934031
bag.oob_score_: 0.934031
Exactly equal: True
Absolute difference: 0.000000000000


The hand-built OOB accuracy was 0.934031 and `bag.oob_score_` was 0.934031; they were exactly equal (`True`, absolute difference 0). The vote counter and `argmax` implement the handout's OOB-only voting and class-0 tie rule.

In [10]:
models = {"tree": tree, "bag": bag, "rf": rf, "lr": lr, "bag_lr": bag_lr}
fold_scores = {
    name: cross_validate(model, X_train, y_train, cv=cv, scoring="accuracy")["test_score"]
    for name, model in models.items()
}
cv_table = pd.DataFrame({name: scores for name, scores in fold_scores.items()},
                        index=[f"fold_{i}" for i in range(1, cv.get_n_splits() + 1)]).T
cv_table["mean"] = [fold_scores[name].mean() for name in cv_table.index]
cv_table["std"] = [fold_scores[name].std() for name in cv_table.index]
print("Accuracy by fold, mean, and population standard deviation:")
display(cv_table.round(4))

paired_differences = {
    "bag - tree": fold_scores["bag"] - fold_scores["tree"],
    "rf - tree": fold_scores["rf"] - fold_scores["tree"],
    "bag_lr - lr": fold_scores["bag_lr"] - fold_scores["lr"],
}
difference_table = pd.DataFrame(
    {name: {**{f"fold_{i + 1}": value for i, value in enumerate(values)},
            "mean": values.mean(), "std": values.std()}
     for name, values in paired_differences.items()}
).T
display(difference_table.round(4))
for name, differences in paired_differences.items():
    result = "inside the noise / no clear gain" if abs(differences.mean()) <= differences.std() else "outside the fold spread"
    print(f"{name}: mean difference={differences.mean():.4f}, std of differences={differences.std():.4f}; {result}.")

Accuracy by fold, mean, and population standard deviation:


,fold_1,fold_2,fold_3,fold_4,fold_5,mean,std
tree,0.8906,0.8891,0.9017,0.9175,0.9095,0.9017,0.0109
bag,0.9414,0.9113,0.9350,0.9444,0.9460,0.9356,0.0128
rf,0.9493,0.9208,0.9445,0.9571,0.9508,0.9445,0.0125
lr,0.9319,0.9192,0.9303,0.9413,0.9508,0.9347,0.0107
bag_lr,0.9366,0.9271,0.9303,0.9429,0.9556,0.9385,0.0101


,fold_1,fold_2,fold_3,fold_4,fold_5,mean,std
bag - tree,0.0507,0.0222,0.0333,0.0270,0.0365,0.0339,0.0097
rf - tree,0.0586,0.0317,0.0428,0.0397,0.0413,0.0428,0.0088
bag_lr - lr,0.0048,0.0079,0.0000,0.0016,0.0048,0.0038,0.0028


bag - tree: mean difference=0.0339, std of differences=0.0097; outside the fold spread.
rf - tree: mean difference=0.0428, std of differences=0.0088; outside the fold spread.
bag_lr - lr: mean difference=0.0038, std of differences=0.0028; outside the fold spread.


Bagging increased tree accuracy by a paired mean of 0.0339 (std of differences 0.0097), so the gain was outside the fold spread. Bagging logistic regression increased accuracy by 0.0038 (std 0.0028), also outside the spread, though the gain was smaller. Averaging bootstrap-fitted models can reduce variance; that effect is much larger for the high-variance tree than for the more stable logistic-regression pipeline.

In [11]:
rf.fit(X_train, y_train)
forest_cv_scores = fold_scores["rf"]
test_accuracy = rf.score(X_test, y_test)
test_se = np.sqrt(test_accuracy * (1 - test_accuracy) / len(y_test))
oob_cv_difference = abs(rf.oob_score_ - forest_cv_scores.mean())
oob_cv_agreement = oob_cv_difference <= forest_cv_scores.std()
print(f"Forest OOB accuracy: {rf.oob_score_:.4f}")
print(f"Forest CV mean +/- std: {forest_cv_scores.mean():.4f} +/- {forest_cv_scores.std():.4f}")
print(f"Absolute OOB-CV difference: {oob_cv_difference:.4f}")
print(f"OOB and CV agree by the handout's fold-spread rule: {oob_cv_agreement}")
print(f"Forest test accuracy: {test_accuracy:.4f} +/- {test_se:.4f} (standard error)")
print(f"Test set size: {len(y_test)}")

Forest OOB accuracy: 0.9489
Forest CV mean +/- std: 0.9445 +/- 0.0125
Absolute OOB-CV difference: 0.0044
OOB and CV agree by the handout's fold-spread rule: True
Forest test accuracy: 0.9524 +/- 0.0066 (standard error)
Test set size: 1051


The forest OOB score (0.9489) and CV mean (0.9445 ± 0.0125) differ by 0.0044, within the CV fold spread, so they agree by the handout's rule. Test accuracy was 0.9524 ± 0.0066 SE; it is 0.0035 above OOB and 0.0079 above the CV mean (about 1.2 test standard errors).